# Experiment 14 — Controlled state sweep

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Pick a small set of canonical negotiation states (e.g. round 4,
8, 13 anchors per policy), save them as a JSON spec, and have a follow-up
data-generation script replay each anchor 10 times. This notebook builds
the spec and — if the results file is available — plots response surfaces.

In [ ]:
import json

anchors = []
for p in POLICIES:
    sub = turns[(turns['policy'] == p) & (~turns['parse_fail']) &
                (turns['round'].isin([4, 8, 13]))]
    for _, g in sub.groupby(['vehicle','round']):
        # pick the run with median B_norm
        g_sorted = g.sort_values('B_norm')
        pick = g_sorted.iloc[len(g_sorted)//2]
        anchors.append({'policy': p, 'vehicle': str(pick['vehicle']),
                        'car_name': str(pick['car_name']),
                        'round': int(pick['round']),
                        'run_id': int(pick['run_id'])})
anchor_spec = {'anchors': anchors, 'replays_per_anchor': 10}
Path('exp14_anchor_spec.json').write_text(json.dumps(anchor_spec, indent=2))
print(f'Saved {len(anchors)} anchors to exp14_anchor_spec.json')
print('Run your data-generation worker against this spec to produce')
print('exp14_sweep_results.json, then re-run this notebook.')